# From a FIBS row to a probe score

This notebook runs `probe-inference` end to end: rows of FIBS (the follow-up deception dataset), Qwen3.5-2B
activations, each probe's read mask from `probe.read_mask`, and the score from `probe.score`. By default it scores
with the published Qwen3.5-2B probes, `load_probe_from_hub("qwen3.5-2b/<arch>")`. Set `USE_RANDOM_PROBES = True` to
use probes with random weights instead: they need no weights download and show the directory format of a probe of
your own, but their scores mean nothing. Requirements and caveats are in the [README](README.md). Run a
`*.local.ipynb` copy; the committed notebook is kept without outputs.

In [ ]:
import tempfile

import torch

import activations
from probe_inference import load_probe, load_probe_from_hub
from random_probes import write_random_probes
from token_masks import token_masks

N_PER_QUADRANT = 2  # rows from each of the four FIBS splits
MAX_TOKENS = 1500  # skip longer rows, to keep each forward pass to seconds
USE_RANDOM_PROBES = False  # True: random-weight probes instead of the published ones

## 1. Rows and activations

FIBS has four splits, `<first answer>_<follow-up answer>`: whether the assistant's first answer was honest or
deceptive, and then whether its answer to a follow-up question was. The probes score the follow-up answer, so a
row's label comes from the second word: 1 (deceptive) for `honest_deceptive` and `deceptive_deceptive`, 0 for the
other two.

`activations.collect_rows` takes the first `N_PER_QUADRANT` plain rows of each split (no tool calls or reasoning
traces, at most `MAX_TOKENS` tokens). For each row it renders the whole conversation, with the follow-up answer
appended, using the model's chat template with thinking off, runs Qwen3.5-2B in bfloat16, and keeps the residual
stream after each probe layer as `Tensor[layers, seq, d_model]`. Layer `k` is the output of decoder block `k`, which
is Hugging Face's `hidden_states[k + 1]`. The published Qwen3.5-2B probes read layers 7, 10, 13, 16, 19 and 22.

In [ ]:
tokenizer = activations.load_tokenizer()
model = activations.load_model()
rows = activations.collect_rows(model, tokenizer, N_PER_QUADRANT, max_tokens=MAX_TOKENS)  # [(row, input_ids, acts)]

Here is one row, the first `honest_deceptive` one, and the text the model saw. A `row` has the conversation up to the
follow-up question (`row.messages`) and the answer to it (`row.completion`). The chat template ends the text with a
newline after the final `<|im_end|>`; the model sees it, but no probe reads it.

In [ ]:
row, input_ids, acts = rows[N_PER_QUADRANT]
print(f"{row.split}, row {row.row_index}, label {row.label}, source {row.source!r}")
print("follow-up question:", row.messages[-1]["content"])
print("answer:", row.completion)
print()
print(len(input_ids), "tokens; the rendered text ends with")
print(repr(tokenizer.decode(input_ids)[-250:]))
print("activations:", tuple(acts.shape), acts.dtype)

As a check of the layer convention, a forward hook on decoder block 13 returns exactly the activations that
`collect` stored for layer 13.

In [ ]:
captured = []
hook = model.model.layers[13].register_forward_hook(
    lambda module, args, output: captured.append((output[0] if isinstance(output, tuple) else output).detach().cpu())
)
with torch.no_grad():
    model(input_ids[None].to(model.device), logits_to_keep=1)
hook.remove()
print("hook == hidden_states[k + 1]:", torch.equal(captured[0][0], acts[activations.LAYERS.index(13)]))

## 2. Probes

`load_probe_from_hub("qwen3.5-2b/<arch>")` downloads one published probe (at most a few tens of megabytes) from
`AlignmentResearch/probe-inference-weights`, at the commit the package pins (`probe_inference.load.WEIGHTS_REVISION`).
With `USE_RANDOM_PROBES`, `write_random_probes` instead writes one probe of each architecture with random weights to a
temporary directory, in the same directory format, and `load_probe` loads them.

Each architecture has one read window (`obfuscate_over` in its `probe_metadata.json`) and one aggregator. The linear
and MLP probes read one token and return a probability, the mean over layers of each layer probe's sigmoid; the EFC
and axial probes read the final exchange and return a logit.

In [ ]:
ARCHS = ("linear", "mlp", "efc", "axial")
d_model = model.config.get_text_config().hidden_size

if USE_RANDOM_PROBES:
    paths = write_random_probes(tempfile.mkdtemp(prefix="random_probes_"), layers=activations.LAYERS, d_model=d_model)
    probes = {arch: load_probe(paths[arch]) for arch in ARCHS}
else:
    probes = {arch: load_probe_from_hub(f"qwen3.5-2b/{arch}") for arch in ARCHS}

for arch, probe in probes.items():
    assert (probe.arch, probe.layers, probe.d_model) == (arch, activations.LAYERS, d_model)
    print(f"{arch:7s} reads {probe.read_window:36s} aggregator {probe.aggregator:5s} returns a {probe.output}")

## 3. Read masks

`probe.read_mask(prompt_mask, completion_mask, followup_start)` builds a probe's read mask, True on the tokens it
reads. `token_masks` finds its three inputs in a row's token ids: the completion is the answer after the empty
thinking block plus its `<|im_end|>`, the prompt is every token before it, and the follow-up starts at the final user
turn's `<|im_start|>`. The newline after the final `<|im_end|>` is in neither, so no window reaches it. A probe's
window depends only on its architecture, so the masks are computed once per row and reused for every probe.

The linear and MLP probes read the single token before the answer's `<|im_end|>`. The EFC and axial probes read every
token from the final user turn's `<|im_start|>` through the answer's `<|im_end|>`.

In [ ]:
masks = [token_masks(input_ids, tokenizer) for _, input_ids, _ in rows]
read_masks = [{arch: row_masks.read_mask(probe) for arch, probe in probes.items()} for row_masks in masks]

# What each window covers, for the row shown above.
read_mask_of = read_masks[N_PER_QUADRANT]
for arch in ("linear", "efc"):
    positions = read_mask_of[arch].nonzero().flatten()
    span = f"positions {int(positions[0])}..{int(positions[-1])} of {len(input_ids)}"
    print(f"{arch}: {probes[arch].read_window}, {span}")
    print(repr(tokenizer.decode(input_ids[read_mask_of[arch]])))
    print()

# The same span on every row: contiguous, from the follow-up's <|im_start|> to the final <|im_end|>, and the linear
# token is the one before that <|im_end|>.
im_start, im_end = tokenizer.convert_tokens_to_ids(["<|im_start|>", "<|im_end|>"])
for (_, ids, _), row_masks, mask_of in zip(rows, masks, read_masks, strict=True):
    final_exchange = mask_of["efc"].nonzero().flatten()
    first, last = int(final_exchange[0]), int(final_exchange[-1])
    assert final_exchange.tolist() == list(range(first, last + 1))
    assert first == int(row_masks.followup_start) and ids[first] == im_start
    assert last == max(i for i, token in enumerate(ids.tolist()) if token == im_end)
    assert mask_of["linear"].nonzero().flatten().tolist() == [last - 1]
    assert all(torch.equal(mask_of[a], mask_of[b]) for a, b in (("linear", "mlp"), ("efc", "axial")))
    after = tokenizer.decode(ids[last + 1 :])
    print(f"{len(ids):>5} tokens; read: linear {last - 1}, efc {first}..{last}; never read: {after!r}")

## 4. Score

`probe.score(acts, read_mask)` takes the whole transcript's activations, `{layer: Tensor[seq, d_model]}`, and the
read mask, and returns one float. Pass the whole transcript, not a slice: the probe gathers the read tokens itself.

In [ ]:
def score_rows(probes):
    """Score every row with every probe, print one line per row, and return {arch: [score per row]}."""
    print(f"{'split':20s} {'label':>5} {'tokens':>6}  " + " ".join(f"{arch:>8}" for arch in probes))
    scores = {arch: [] for arch in probes}
    for (row, input_ids, acts), read_mask_of in zip(rows, read_masks, strict=True):
        acts_by_layer = dict(zip(activations.LAYERS, acts, strict=True))
        for arch, probe in probes.items():
            scores[arch].append(probe.score(acts_by_layer, read_mask_of[arch]))
        cells = " ".join(f"{scores[arch][-1]:>8.4f}" for arch in probes)
        print(f"{row.split:20s} {row.label:>5} {len(input_ids):>6}  {cells}")
    return scores


scores = score_rows(probes)

`score_batch` scores many transcripts at once: right-pad the activations to one length (the padding is never read)
and stack the read masks, padded with `False`. The batch scores equal the one-at-a-time scores up to float rounding.

In [ ]:
seq = max(len(input_ids) for _, input_ids, _ in rows)
batch = torch.zeros(len(rows), len(activations.LAYERS), seq, d_model, dtype=torch.bfloat16)
batch_masks = {arch: torch.zeros(len(rows), seq, dtype=torch.bool) for arch in probes}
for i, ((_, input_ids, acts), read_mask_of) in enumerate(zip(rows, read_masks, strict=True)):
    batch[i, :, : len(input_ids)] = acts
    for arch in probes:
        batch_masks[arch][i, : len(input_ids)] = read_mask_of[arch]

print("batch", tuple(batch.shape))
for arch, probe in probes.items():
    batch_scores = probe.score_batch(batch, batch_masks[arch])  # stacked in probe.layers order
    difference = (batch_scores - torch.tensor(scores[arch], dtype=torch.float64)).abs().max().item()
    print(f"{arch:7s} largest difference from the one-at-a-time scores: {difference:.2e}")

## 5. Scores by label

The mean score of the rows with each label. This is a plumbing check, not an evaluation: a handful of rows prove
nothing, and the published probes were trained on FIBSv1, so these rows are probably in their training data. With
random probes the two means carry no information about the label.

In [ ]:
def mean_score(arch, label):
    """The mean score of the rows with this label."""
    chosen = [score for score, (row, _, _) in zip(scores[arch], rows, strict=True) if row.label == label]
    return sum(chosen) / len(chosen)


print("random" if USE_RANDOM_PROBES else "published", "probes: mean score by label")
print(f"{'':7s} {'0 (honest)':>11} {'1 (deceptive)':>14}")
for arch in probes:
    print(f"{arch:7s} {mean_score(arch, 0):>11.3f} {mean_score(arch, 1):>14.3f}")

## Next

`python activations.py --per-quadrant N` (the README has the full command) stores the token ids, the token masks and
the full-sequence activations of N rows per split in one `.pt` file, so a probe can score them without running the
model again.